# MobSys lab — parsing a 5G NR cell search

In this lab you will discover the configuration of the 5G cells around a phone by
looking at the very first messages a UE decodes: the **MIB** (on PBCH) and **SIB1**
(on PDSCH, scheduled by a PDCCH in CORESET#0).

The trace was taken outdoors in Mouans-Sartoux (06370, France) with a
**Samsung Galaxy S23 Ultra** (Snapdragon 8 Gen 2) using Qualcomm's diagnostic
interface. The raw baseband log (`.hdf`) was converted into a `.pcap` with
[scat](https://github.com/fgsect/scat), so that the RRC messages can be decoded by
Wireshark. You can do every question either:

* in **Wireshark** (open `../data/cell_search.pcap`, see the README for the plugin), or
* in this **notebook**, which calls `tshark` (Wireshark's command-line version) for you.

Useful references (all free on [3gpp.org](https://www.3gpp.org/specifications)):
TS 38.104 (ARFCN, §5.4.2), TS 38.211 (physical channels), TS 38.213 (§10 PDCCH,
§13 CORESET#0 tables), TS 38.214 (time-domain allocation, §5.1.2 / §6.1.2),
TS 38.331 (RRC ASN.1).

## 0. Setup

Run the cell below. It finds the helper module `nrlab.py`, checks that `scat` and
`tshark` are installed, and (if you have the raw `.hdf` trace from Moodle next to
this notebook) re-creates the pcap from it. Otherwise it uses the pcap provided in
`../data/`.

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'nrlab.py').exists())
sys.path.insert(0, str(LAB))
import nrlab
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160)

nrlab.check_tools()

HDF = LAB / 'cell_search.hdf'            # raw trace from Moodle, if you have it
PCAP = LAB.parent / 'data' / 'cell_search.pcap'
if HDF.exists():
    PCAP = nrlab.run_scat(HDF, LAB / 'cell_search.pcap')
print('Using', PCAP)

## 1. The RRC messages of a cell search

`nrlab.packets()` returns one row per message with the information scat took from
the Qualcomm log header:

| column | meaning |
|---|---|
| `arfcn` | NR-ARFCN of the SSB on which the message was received |
| `pci` | Physical Cell Identity |
| `comment` | information about the UE |
| `message` | RRC message type |

In Wireshark the same values are in the *GSMTAP* part of each packet
(`gsmtapv3.channel_number`, `gsmtapv3.multiplex`, `gsmtapv3.packet_comment`);
right-click → *Apply as Column*.

In [ ]:
df = nrlab.packets(PCAP)
df

### Question 1


You see two kinds of RRC messages, *BCCH-BCH* (MIB) and *BCCH-DL-SCH* (SIB1).
Explain what each corresponds to: which physical channel carries it, how the UE
finds it, and why both are needed.

In [ ]:
# (no code needed)

*Your answer:*

### Question 2


Which 3GPP **release** of the RRC protocol does the Galaxy S23 Ultra implement?
(look at the `comment` column, or `gsmtapv3.packet_comment` in Wireshark)

In [ ]:
df['comment'].unique()

*Your answer:*

### Question 3


Which **frequencies** did the UE find during the cell search? Write a function
`arfcn_to_mhz(n)` that converts an NR-ARFCN into a frequency, using TS 38.104
§5.4.2.1:

$F_\mathrm{REF} = F_\mathrm{REF-Offs} + \Delta F_\mathrm{Global}\,(N_\mathrm{REF} - N_\mathrm{REF-Offs})$

| $N_\mathrm{REF}$ range | $\Delta F_\mathrm{Global}$ | $F_\mathrm{REF-Offs}$ | $N_\mathrm{REF-Offs}$ |
|---|---|---|---|
| 0 – 599999 | 5 kHz | 0 MHz | 0 |
| 600000 – 2016666 | 15 kHz | 3000 MHz | 600000 |
| 2016667 – 3279165 | 60 kHz | 24250.08 MHz | 2016667 |

and make a table: ARFCN, frequency, number of different PCIs, messages seen.
You can check your function with <https://5g-tools.com/5g-nr-arfcn-calculator/>.
For more on the ARFCN see TS 38.101-1 (UE) and TS 38.104 (gNB).

In [ ]:
def arfcn_to_mhz(n):
    ...  # your code here

*Your answer:*

### Question 4


Can you identify the **operators** of these frequencies? Use
[cartoradio](https://www.cartoradio.fr/index.html#/cartographie/lonlat/6.967469/43.619687):
click on the sites around the measurement point to see the operators present,
their technologies (2G/3G/4G/5G), bands and antenna directions.

In [ ]:
# (no code needed)

*Your answer:*

### Question 5


Why do some operators provide 5G services at **low frequencies** (700 MHz, 2.1 GHz)?

In [ ]:
# (no code needed)

*Your answer:*

### Question 6


Why might we see SIB1 (BCCH-DL-SCH) only at 3.5 GHz, and not on the lower
frequencies of the same operator? Look at the MIB fields `cellBarred` and
`ssb-SubcarrierOffset` (k_SSB) of each carrier; TS 38.213 §13 explains what
happens when $k_\mathrm{SSB} > 23$ in FR1.

In [ ]:
mib = nrlab.packets(PCAP, barred='nr-rrc.cellBarred', kssb='nr-rrc.ssb_SubcarrierOffset',
                    scs='nr-rrc.subCarrierSpacingCommon',
                    coreset0='nr-rrc.controlResourceSetZero', ss0='nr-rrc.searchSpaceZero')
mib[mib.message == 'MIB']

*Your answer:*

### Question 7


One information element in the MIB is `subCarrierSpacingCommon`.

1. Why is there no ambiguity in a value like `scs30or120` or `scs15or60`?
   (Hint: the UE knows whether it is in FR1 or FR2.)
2. Why would the lower frequencies be forced to choose `scs15or60`? (Hint: look at
   what an operator has deployed in those bands on cartoradio, e.g. 700 MHz, 2.1 GHz.)

In [ ]:
# use the mib table from Question 6

*Your answer:*

## 2. SIB1

Now look at the BCCH-DL-SCH packets, which contain `SystemInformationBlockType1`
(SIB1). `nrlab.show(PCAP, frame)` prints the decoded message (like Wireshark's
packet-details pane); `nrlab.show(PCAP, frame, grep='text')` keeps only the lines
containing `text`, and `nrlab.field(PCAP, frame, 'nr-rrc.xxx')` returns all values
of one field. Field names are the ASN.1 names with `-` replaced by `_`
(`ra-ResponseWindow` → `nr-rrc.ra_ResponseWindow`); in Wireshark, select a field
and look at the status bar to see its name.

**Careful:** for ENUMERATED fields, `field()` returns the *index* of the value, not
the value: `ra-ResponseWindow: sl80 (7)` comes back as `'7'`. Use `show()` to see
the names, and the ASN.1 in TS 38.331 to map the index.

In [ ]:
sib1_frames = df[df.message == 'SIB1'][['frame', 'arfcn', 'pci']]
sib1_frames

In [ ]:
nrlab.show(PCAP, int(sib1_frames.frame.iloc[0]))

### Question 8


At 3.5 GHz, how many operators transmit the SSB, and how many transmit SIB1?
What does this mean for those which only transmit the SSB?

In [ ]:
# (no code needed)

*Your answer:*

### Question 9


In SIB1, `cellAccessRelatedInfo` provides the basic identifiers of the cell, in
particular the **PLMN** identities (MCC + MNC). 208 is the MCC of France, which
uses 2-digit MNCs. An operator can broadcast more than one PLMN on a given gNB
(which is the case for one of them here). Verify your answer to Question 4 for the
operators which send SIB1 by looking up their PLMNs, and also report the tracking
area code.

In [ ]:
for f in sib1_frames.frame:
    print(f, nrlab.field(PCAP, int(f), 'nr-rrc.MCC_MNC_Digit'), nrlab.field(PCAP, int(f), 'nr-rrc.trackingAreaCode'))

*Your answer:*

## 3. ServingCellConfigCommon

`servingCellConfigCommon` is the main information element of SIB1. It defines

* spectrum allocations for downlink and uplink,
* the **initial bandwidth part** used for common signalling,
* DL / UL time allocations for PDSCH / PUSCH,
* the random-access configuration,
* the PDCCH configuration (CORESETs and search spaces),
* PUCCH, BCCH and PCCH configurations, SSB configuration and the TDD pattern.

We start with what the UE uses to find SIB1 itself: CORESET#0.

![frequency layout](../figures/coreset0_layout.svg)

The MIB gives `ssb-SubcarrierOffset` = $k_\mathrm{SSB}$ (in FR1, units of 15 kHz
subcarriers: the offset between the start of the SSB and the start of the common
resource block that contains it) and `pdcch-ConfigSIB1`, which contains

* `controlResourceSetZero`: an index into one of TS 38.213 Tables 13-1 … 13-10
  (chosen from the SSB/PDCCH subcarrier spacings and the band's minimum channel
  bandwidth). The row gives the **number of RBs** and **number of symbols** of
  CORESET#0 and its **offset** in RBs below the first RB that overlaps the SSB.
* `searchSpaceZero`: an index into Table 13-11 (FR1), which gives the slots and
  symbols where the UE looks for the PDCCH scheduling SIB1 (Type0-PDCCH CSS).

### Question 10


Locate the frequency grid of CORESET#0 (where the PDCCH and PDSCH of SIB1 are)
for the 3.5 GHz SSBs: which table of TS 38.213 applies (n78 has a minimum
channel bandwidth of 10 MHz, SSB and PDCCH at 30 kHz), what is the CORESET#0
size in RBs, and where does it start relative to the SSB?

In [ ]:
# controlResourceSetZero and searchSpaceZero are in the MIB table of Question 6

*Your answer:*

### Question 11


1. Over how many OFDM symbols can the PDCCH of SIB1 be located (size of CORESET#0)?
2. Using `searchSpaceZero` and TS 38.213 Table 13-11, find in which **slots** and
   **symbols** the UE monitors the Type0-PDCCH for each SSB index $i$. Recall
   $n_0 = (O \cdot 2^{\mu} + \lfloor i \cdot M \rfloor) \bmod N^{\mathrm{frame},\mu}_{\mathrm{slot}}$,
   the UE monitors slots $n_0$ and $n_0+1$, in frames with
   $\lfloor (O\cdot 2^\mu + \lfloor iM\rfloor)/N^{\mathrm{frame},\mu}_{\mathrm{slot}}\rfloor \bmod 2$
   = SFN mod 2.

In [ ]:
# your code here

*Your answer:*

### Question 12


A PDCCH is made of $L$ **CCEs** (aggregation level $L \in \lbrace 1,2,4,8,16\rbrace$), one CCE
= 6 REGs, one REG = 1 RB × 1 symbol (so a CCE has 72 REs, 54 for data and 18 for
DMRS; in a 1-symbol CORESET a CCE is 6 consecutive RBs). For the
Type0-PDCCH CSS, TS 38.213 Table 10.1-1 defines 4 candidates at $L=4$, 2 at $L=8$
and 1 at $L=16$.

1. How many CCEs does CORESET#0 contain here?
2. How many *distinct* PDCCH candidates can actually carry the DCI of SIB1, at
   each aggregation level? (the CCEs of candidate $m$ start at
   $L\lfloor m\,N_\mathrm{CCE}/(L\,M_L)\rfloor$, TS 38.213 §10.1, for a common search space)
3. Compare with the slide example (48 RBs × 2 symbols). Which DCI format and which
   RNTI does the UE look for?

(With QCAT, the logs *NR5G MAC DCI Info* and *NR5G MAC PDSCH Status* show the
DCI actually decoded by the modem; they are encrypted in the scat version of this trace.)

In [ ]:
# your code here

*Your answer:*

### Question 13


SIB1 also defines further common search spaces in `pdcch-ConfigCommon`
(`commonSearchSpaceList`) and says which one is used for other system
information, for paging and for random access. For each of them give the
CORESET, the monitoring periodicity/offset/duration, the symbols, and the
aggregation levels. Why is the RA search space monitored in every slot?

In [ ]:
# nrlab.show(PCAP, frame, grep=...) may help

*Your answer:*

### Question 14


After SIB1 the UE knows the full frequency grid of the cell. **Point A** is the
centre of subcarrier 0 of common resource block 0. `offsetToPointA` (in SIB1) is
the offset between Point A and the lowest subcarrier of the CRB that overlaps the
SSB, **in RBs of 15 kHz subcarrier spacing for FR1**, and $k_\mathrm{SSB}$ (MIB) the
remaining offset in 15 kHz subcarriers. The NR-ARFCN of the SSB is the frequency of
its subcarrier 120 (the 0th subcarrier of its 11th RB).
`scs-SpecificCarrierList` gives the subcarrier spacing and the carrier bandwidth.

For the SA cells, determine:

1. the subcarrier spacing of the cells,
2. the cell bandwidths in PRBs (check they match the operators' documented n78
   bandwidths using TS 38.101-1 Table 5.3.2-1),
3. the frequency of Point A and of the carrier edges,
4. the real offset of the SSB relative to Point A, in PRBs of the carrier's
   subcarrier spacing.

In [ ]:
# your code here

*Your answer:*

### Question 15


The **initial downlink and uplink BWP** are given by `locationAndBandwidth` in
`initialDownlinkBWP` / `initialUplinkBWP`. It is a resource indication value
(RIV, TS 38.214 §5.1.2.2.2) computed with $N^\mathrm{size}_\mathrm{BWP} = 275$:

* if $(L_\mathrm{RB} - 1) \le \lfloor 275/2 \rfloor$: $\mathrm{RIV} = 275(L_\mathrm{RB} - 1) + RB_\mathrm{start}$
* else: $\mathrm{RIV} = 275(275 - L_\mathrm{RB} + 1) + (275 - 1 - RB_\mathrm{start})$

Write `riv_decode(riv)` returning $(RB_\mathrm{start}, L_\mathrm{RB})$ and give the initial
BWPs of both cells (in CRBs). Compare them with CORESET#0 and with the carrier.

In [ ]:
def riv_decode(riv, n=275):
    ...  # your code here

*Your answer:*

## 4. Time-domain allocations, random access and UL

`pdsch-ConfigCommon` contains `pdsch-TimeDomainAllocationList`. A DCI scheduling a
PDSCH points to one of its entries, which has three parameters:

* `k0`: the slot offset of the PDSCH relative to the slot of the DCI,
* `mappingType`: type A (DMRS position relative to the slot start) or B (mini-slot),
* `startSymbolAndLength`: the start symbol $S$ and length $L$ coded as a **SLIV**
  (TS 38.214 §5.1.2.1):

$$\mathrm{SLIV} = \begin{cases} 14(L-1) + S & \text{if } (L-1) \le 7 \\ 14(14-L+1) + (14-1-S) & \text{otherwise} \end{cases}
\quad\text{with } 0 < L \le 14 - S$$

### Question 16


For the SA cells, decode the possible PDSCH time allocations into real start
symbols and lengths. Write `sliv_decode(sliv)`. Why do none of them start at symbol 0?

In [ ]:
def sliv_decode(sliv):
    ...  # your code here

*Your answer:*

`rach-ConfigCommon` configures the random-access procedure:

* `prach-ConfigurationIndex`: an index into TS 38.211 Table 6.3.3.2-2 (FDD/SUL),
  6.3.3.2-3 (TDD FR1) or 6.3.3.2-4 (FR2), giving the preamble format, the frames
  (SFN mod x = y), the subframes/slots and the symbols of the RACH occasions,
* `msg1-FDM`, `msg1-FrequencyStart`: number of RACH occasions in frequency, and the first PRB,
* `zeroCorrelationZoneConfig`: the cyclic-shift size $N_\mathrm{CS}$,
* power ramping: `preambleReceivedTargetPower`, `powerRampingStep`, `preambleTransMax`,
* `ra-ResponseWindow`, and `ssb-perRACH-OccasionAndCB-PreamblesPerSSB` (SSB ↔ RACH occasion association).

### Question 17


Look up `prach-ConfigurationIndex` of both cells in TS 38.211 Table 6.3.3.2-3
(TDD, FR1). Which preamble format is used, in which frames and subframes are the
RACH occasions, and do they fall in uplink slots of the TDD pattern (Question 22)?
With `ssb-perRACH-Occasion` and the number of SSBs (Question 21), how long does it
take until every SSB beam has had at least one RACH occasion?

In [ ]:
# your code here

*Your answer:*

### Question 18


The UE estimates the path loss from the received SSB power and the signalled
`ss-PBCH-BlockPower`, and sets its PRACH power so that the gNB receives it at
`preambleReceivedTargetPower`. If no RAR (Msg2) arrives within `ra-ResponseWindow`,
it retransmits, adding `powerRampingStep` each time, up to `preambleTransMax`
transmissions.

What is the range of PRACH **received** target powers in each SA cell, and the RAR
window size in slots and in ms? How long can one Msg1 attempt take?

In [ ]:
# your code here

*Your answer:*

### Question 19


`pusch-ConfigCommon` has a time-domain allocation list encoded like the PDSCH one,
where each entry has a $k_2$: the number of slots between the DCI (or RAR) and the
PUSCH. What are the possible scheduling delays $k_2$, and the possible PUSCH
start/length? Relate $k_2$ to the TDD pattern. Also report `msg3-DeltaPreamble`
and `p0-NominalWithGrant`: what do they control?

In [ ]:
# your code here

*Your answer:*

### Question 20


`pucch-ConfigCommon` configures the PUCCH used for HARQ-ACK of Msg4 before the UE
has a dedicated configuration. `pucch-ResourceCommon` is a row of TS 38.213 Table
9.2.1-1. Which PUCCH format, symbols and PRB offset do the commercial networks use?

In [ ]:
# your code here

*Your answer:*

### Question 21


The SSB is configured with `ssb-PositionsInBurst` (a bitmap of the SSB indexes
actually transmitted out of $L_\mathrm{max}$; 8 for n78) and
`ssb-PeriodicityServingCell`. Explain the difference between the operators' SSB
configurations in terms of the number of beams. In which slots and symbols are the
SSBs (n78 uses SSB pattern "Case C": first symbols $\lbrace 2, 8\rbrace + 14n$)?

In [ ]:
# your code here

*Your answer:*

### Question 22


`tdd-UL-DL-ConfigurationCommon` gives the TDD pattern: `dl-UL-TransmissionPeriodicity`,
`nrofDownlinkSlots`, `nrofDownlinkSymbols` (DL symbols at the start of the mixed slot),
`nrofUplinkSlots`, `nrofUplinkSymbols` (UL symbols at the end of the mixed slot).
The guard has $14 - $`nrofDownlinkSymbols`$ - $`nrofUplinkSymbols` symbols.
Write the TDD pattern of the SA cells in the usual form (e.g. DDDSU) and draw it.
Why do all operators in the band use the same pattern?

![TDD pattern](../figures/tdd_pattern.svg)

In [ ]:
# your code here

*Your answer:*

## Going further

* Compare with the second capture, `cell_search_051124`: which cells and SIB1s are
  different?
* Draw the frequency layout of the Orange carrier: Point A, the CRB grid, CORESET#0,
  the SSB and the initial BWP, with the numbers you computed.
* How long does a UE need, at minimum, from the first SSB detection to having SIB1?